Building the PyPSA model that minimises total annual system costs

In [17]:
!pip install pypsa
!pip install atlite
import pypsa
import numpy as np
import pandas as pd
import atlite
import matplotlib.pyplot as plt
from pathlib import Path
import geopandas as gpd
import rasterio
from atlite.gis import ExclusionContainer
import xarray as xr

In [18]:
from google.colab import drive
drive.mount('/content/drive')

#The codes below are copied from the file named 02_land_eligibility_solar.ipynb and 03_land_eligibility_wind.ipynb
BASE = Path("/content/drive/MyDrive/HW4/Data")
RAW = BASE / "RAW"
PROCESSED = BASE / "PROCESSED"

gadm_path = RAW / "gadm" / "gadm_410-levels-ADM_1-AUT.gpkg"
gadm = gpd.read_file(gadm_path)
regions_m = gadm.to_crs("EPSG:3035").set_index("NAME_1")

#Now PyPSA model is built
n = pypsa.Network()
n.set_snapshots(pd.date_range("2019-01-01", "2019-12-31 23:00", freq="3h")) #3 hourly time steps

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Adding Buses (Buses represent regions)



In [19]:
region_centroids = gpd.read_file(PROCESSED / "austria_region_centroids.geojson")
region_centroids = region_centroids.to_crs("EPSG:3035")
region_centroids = region_centroids.set_index("region")

for region in regions_m.index:
    n.add(
        "Bus",
        region,
        x=region_centroids.loc[region, "x"], #locations of centroids are added
        y=region_centroids.loc[region, "y"],
    )

Adding Loads

In [21]:
regional_load = pd.read_csv(PROCESSED / "regional_load.csv", index_col=0, parse_dates=True) #we need to first save regional_load.csv under PROCESSED

regional_load_3h = regional_load.resample("3h").mean()
regional_load_3h = regional_load_3h.reindex(n.snapshots)

for region in regions_m.index:
    n.add(
        "Load",
        f"load {region}",
        bus=region,
        p_set=regional_load_3h[region],
    )

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/HW4/Data/PROCESSED/regional_load.csv'

 Technology Assumptions (costs, efficiencies, lifetimes, etc.)

In [23]:
cost_year = 2030 #could be another year too
costs_url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{cost_year}.csv"
costs = pd.read_csv(costs_url, index_col=[0, 1])

discount_rate = 0.07 #given

def annuity(r, lifetime):
    return r / (1 - (1 + r) ** (-lifetime))

#the code below is from claude, i could not check them yet bc time is late but we can check before writing further
def get_cost(tech, parameter, default=0.0):
    try:
        return costs.loc[(tech, parameter), "value"]
    except KeyError:
        return default

power_techs = ["CCGT", "OCGT", "coal", "oil", "biomass", "hydro", "onwind", "solar-utility"]

fuel_row = {
    "CCGT": "gas", "OCGT": "gas", "coal": "coal", "oil": "oil", "biomass": "biomass",
    "hydro": None, "onwind": None, "solar-utility": None,
}

cost_table = pd.DataFrame(index=power_techs)
cost_table["investment"] = [get_cost(t, "investment") for t in power_techs]       # EUR/kW
cost_table["FOM"] = [get_cost(t, "FOM") for t in power_techs]                     # %/yr
cost_table["VOM"] = [get_cost(t, "VOM") for t in power_techs]                     # EUR/MWh
cost_table["efficiency"] = [get_cost(t, "efficiency", default=1.0) for t in power_techs]
cost_table["lifetime"] = [get_cost(t, "lifetime", default=25) for t in power_techs]
cost_table["fuel_cost"] = [get_cost(fuel_row[t], "fuel") if fuel_row[t] else 0.0 for t in power_techs]
cost_table["co2_intensity"] = [get_cost(fuel_row[t], "CO2 intensity") if fuel_row[t] else 0.0 for t in power_techs]

cost_table["capital_cost"] = (
    (annuity(discount_rate, cost_table["lifetime"]) + cost_table["FOM"] / 100)
    * cost_table["investment"] * 1000  # EUR/kW -> EUR/MW
)  # EUR/MW/yr
cost_table["marginal_cost"] = cost_table["VOM"] + cost_table["fuel_cost"] / cost_table["efficiency"]
cost_table["co2_emissions"] = cost_table["co2_intensity"] / cost_table["efficiency"]  # tCO2/MWh_el

cost_table.loc["biomass", "co2_emissions"] = 0.0

cost_table


,investment,FOM,VOM,efficiency,lifetime,fuel_cost,co2_intensity,capital_cost,marginal_cost,co2_emissions
CCGT,1108.7166,3.3494,5.6104,0.580,25.0,28.4158,0.1980,132274.898698,54.603159,0.341379
OCGT,581.3949,1.7795,6.0111,0.410,25.0,28.4158,0.1980,60235.719324,75.317929,0.482927
coal,4812.0244,1.3100,4.1005,0.356,40.0,7.8202,0.3361,423983.326123,26.067354,0.944101
oil,458.1805,2.4630,8.0148,0.350,25.0,43.6295,0.2571,50601.691400,132.670514,0.734571
biomass,2950.7891,4.5269,0.0000,0.468,30.0,9.3506,0.0000,371372.752857,19.979915,0.000000
hydro,2888.5279,1.0000,0.0000,0.900,80.0,0.0000,0.0000,231987.992780,0.000000,0.000000
onwind,1383.3059,1.2167,1.8033,1.000,30.0,0.0000,0.0000,128306.330322,1.803300,0.000000
solar-utility,482.4785,2.4757,0.0000,1.000,40.0,0.0000,0.0000,48135.017035,0.000000,0.000000
